In [0]:
"""
02_operation_events.py

Creates the Silver Operation Events table.

Input:
    parsed_events

Output:
    operation_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import col


# ============================================================
# Operation Events
# ============================================================

@dlt.table(
    name="operation_events",
    comment="Validated manufacturing operation events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect_or_drop(
    "valid_machine_id",
    "machine_id IS NOT NULL",
)

@dlt.expect_or_drop(
    "valid_work_order",
    "work_order_id IS NOT NULL",
)

@dlt.expect_or_drop(
    "valid_serial_number",
    "serial_number IS NOT NULL",
)

@dlt.expect(
    "positive_force",
    "actual_force_kn > 0",
)

@dlt.expect(
    "positive_cycle_time",
    "cycle_time_sec > 0",
)

def operation_events():

    df = spark.readStream.table("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only operation events
        # -----------------------------------------

        .filter(
            col("event_type") == "OPERATION_COMPLETED"
        )

        # -----------------------------------------
        # Select business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "hall_id",
            "line_id",
            "machine_id",

            "execution_id",
            "work_order_id",
            "serial_number",

            "operator_id",

            "product_code",

            "source_system",

            "correlation_id",

            "silver_processing_timestamp",

            "payload.operation_number",
            "payload.operation_name",

            "payload.department",

            "payload.machine_name",
            "payload.machine_type",

            "payload.station_code",
            "payload.station_type",

            "payload.line_name",
            "payload.hall_name",

            "payload.tool_name",
            "payload.tool_type",

            "payload.operator_name",
            "payload.skill_level",

            "payload.target_force_kn",
            "payload.actual_force_kn",
            "payload.force_deviation_kn",

            "payload.displacement_mm",

            "payload.cycle_time_sec",

            "payload.quality_result",

        )

    )